In [ ]:
import cv2
import random
import insightface
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
from insightface.app import FaceAnalysis

# run

In [ ]:
# img to embedding function

def img_to_embedding(img_path):

    # read img
    read_img = cv2.imread(img_path)

    # init face analysis app
    app = FaceAnalysis()

    # to prepare images 
    app.prepare(ctx_id=0)
    try:
        faces = app.get(read_img)[0]['embedding']
        return faces
    
    except Exception as e:
        print(f"Error in embedding function for {img_path}: {e}")
        return None

# start live demo

In [ ]:
# load sabrina embedding pickle 
path = "/mnt/nas2/sabrina/face-gen/embeddings_sabrina.pkl"

df_sabrina = pd.read_pickle(path)

is_array_col = df_sabrina['embedding'].apply(lambda x: isinstance(x, np.ndarray))

# convert back to float32
df_sabrina['embedding'] = df_sabrina['embedding'].apply(lambda x: x.astype(np.float32))
print(df_sabrina.shape)


In [ ]:
# query on live sabrina
query_path =  '/home/sho/insightface/gsi/img/demo.png'
query_sabrina = img_to_embedding(query_path)

In [ ]:
# brute force search
detector = insightface.model_zoo.get_model('/home/sho/.insightface/models/buffalo_l/w600k_r50.onnx')

def brute_force(df, query):
    # sim scores
    similarities = []

    start_time = datetime.now()
    
    for i in range(len(df)):
        emb = df['embedding'].iloc[i]
        sim = detector.compute_sim(query, emb)
        similarities.append(sim)

    # top 10
    top_k_indices = np.argsort(similarities)[::-1][:10]
    top_k_scores = [similarities[i] for i in top_k_indices]
    top_k_ids = [int(df.iloc[i]['ID']) for i in top_k_indices]

    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    print(f"search time: {duration} seconds")

    print(f"top k person ID:{top_k_ids}")

    return top_k_indices, top_k_scores

In [ ]:
# hnsw search
import hnswlib

def hnsw_search(df, query):

    embedding_matrix = np.stack(df['embedding'].values)
    dim = embedding_matrix.shape[1]

    query_embedding = query.reshape(1, -1)

    start_time = datetime.now()
    # Build the index
    index_drop = hnswlib.Index(space='cosine', dim=dim)
    index_drop.init_index(max_elements=len(embedding_matrix), ef_construction=200, M=16)

    # Add embeddings to the index
    index_drop.add_items(embedding_matrix, np.arange(len(embedding_matrix)))
    
    top_k = 10
    labels, distances = index_drop.knn_query(query_embedding, k=top_k)

    top_k_indices = labels[0]
    top_k_scores = distances[0]
    top_k_ids = [df.iloc[idx]['ID'] for idx in top_k_indices]

    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    print(f"search time: {duration} seconds")

    return top_k_indices, top_k_scores


In [ ]:
# fvs search
import fvs_face_helper as fvs

def fvs_search(df, query):

    query = query.reshape((1, 512))

    start_time = datetime.now()
    response = fvs.face_search("d4b33d5c-0719-411b-ace4-8b5ff8b4004c", query, 10, True )
    #print("response=", response)

    top_k_indices = response.indices[0]
    top_k_scores = response.distance[0]
    top_k_scores = [float(x) for x in top_k_scores]
    top_k_ids = [int(df.iloc[i]['ID']) for i in top_k_indices]
    top_k_names = [df.iloc[i]['name'] for i in top_k_indices]

    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    print(f"search time: {duration} seconds")

    print(f"top 1 name:{top_k_names[0]}")

    # 768

    return top_k_indices, top_k_scores


# show results

In [ ]:
# plot
def plot_results(query_path, top_k_indices, top_k_scores, method):
    rows = 2
    cols = 5
    fig = plt.figure(figsize=(16, 10))

    # plot query
    query_img = cv2.imread(query_path)
    query_img_rgb = query_img[:, :, ::-1]

    ax = fig.add_subplot(rows + 1, cols, 1)
    ax.imshow(query_img_rgb)
    ax.set_title("query image")
    ax.axis("off")

    for i in range(1, cols):
        ax = fig.add_subplot(rows + 1, cols, i + 1)
        ax.axis("off")

    for idx, i in enumerate(top_k_indices):
        fp = df_sabrina.iloc[i]['filepath']
        read_img = cv2.imread(fp)
        read_img_rgb = read_img[:, :, ::-1]

        sim_score = top_k_scores[idx]

        ax = fig.add_subplot(rows + 1, cols, cols + idx + 1)  # start from row 2
        ax.imshow(read_img_rgb)
        if method == 'brute_force':
            title = f"ID: {df_sabrina.iloc[i]['ID']} | Sim (cosine sim): {sim_score:.4f}"
        elif method == 'hnsw':
            title = f"ID: {df_sabrina.iloc[i]['ID']} | Sim (cosine dist): {sim_score:.4f}"
        elif method == 'fvs':
            title = f"ID: {df_sabrina.iloc[i]['ID']} | Sim (hamming): {sim_score:.4f}"
        else:
            title = "method does not exist"
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# show whole gallery
import fiftyone as fo
import os

# in a local terminal window, make sure to setup an SSH tunnel like so
# ssh -N -L 5151:localhost:5151 <user>@192.168.99.33


In [ ]:
local_paths = []
for i in range(df_sabrina.shape[0]):
    fpath = df_sabrina.iloc[i]['filepath']
    if not os.path.exists(fpath):
        raise Exception("path does not exist")
    local_paths.append(fpath)

In [ ]:
# Create FO dataset
samples = []
for filepath in local_paths:
    parent = os.path.basename( os.path.dirname(filepath))
    sample = fo.Sample(filepath=filepath)
    sample["ground_truth"] = fo.Classification(label=parent)
    samples.append(sample)

In [ ]:
# Create dataset
dataset = fo.Dataset("face-rec-demo3")
ids = dataset.add_samples(samples)

In [ ]:
print("starting...")
session = fo.launch_app(dataset, auto=False) #address="0.0.0.0", port=5151)
print("end")

In [ ]:
session.open_tab()

# start here

1. show gallery
2. show "sabrina" in gallery
3. show query image

In [ ]:
# show query image

query_img = cv2.imread(query_path)
query_img_rgb = query_img[:, :, ::-1]
plt.imshow(query_img_rgb)
plt.show()


In [ ]:
# fvs results
fvs_ind, fvs_scores = fvs_search(df_sabrina, query_sabrina)
plot_results(query_path, fvs_ind, fvs_scores, 'fvs')


In [ ]:
# hnsw results
hnsw_ind, hnsw_scores = hnsw_search(df_sabrina, query_sabrina)
plot_results(query_path, hnsw_ind, hnsw_scores, 'hnsw')


In [ ]:
# brute force results
bf_ind, bf_scores = brute_force(df_sabrina, query_sabrina)
plot_results(query_path, bf_ind, bf_scores, 'brute_force')